In [1]:
!pip install -q FlagEmbedding openpyxl

In [2]:
import torch
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition


In [5]:
from google.colab import files
import os

TOP10_PATH = "/content/SANAD_Experiment_B_Top10_for_Reranking.csv"

print("Upload: SANAD_Experiment_B_Top10_for_Reranking.csv")
uploaded = files.upload()

for name, data in uploaded.items():
    if "Top10_for_Reranking" in name:
        with open(TOP10_PATH, "wb") as f:
            f.write(data)

assert os.path.exists(TOP10_PATH), " B"
print("Using:", TOP10_PATH)

Upload: SANAD_Experiment_B_Top10_for_Reranking.csv


Saving SANAD_Experiment_B_Top10_for_Reranking.csv to SANAD_Experiment_B_Top10_for_Reranking.csv
Using: /content/SANAD_Experiment_B_Top10_for_Reranking.csv


In [6]:
import re
import numpy as np
import pandas as pd

TOP_K = 10

# B's column names -> names we use in this notebook
B_COLS = {
    "candidate_rank": "retrieval_rank",
    "doc_index": "doc_idx",
    "candidate_law": "law_name",
    "candidate_article": "article_number",
    "candidate_text": "text",
}
TEXT_COLS = ["query_id", "query_type", "query", "gold_law", "gold_article",
             "selected_retrieval", "candidate_law", "candidate_article", "candidate_text"]

top10_df = pd.read_csv(TOP10_PATH, encoding="utf-8-sig", keep_default_na=False,
                       dtype={c: str for c in TEXT_COLS})

REQUIRED = ["query_id", "query_type", "query", "gold_law", "gold_article",
            "selected_retrieval"] + list(B_COLS)
missing_cols = [c for c in REQUIRED if c not in top10_df.columns]
assert not missing_cols, f"B's file is missing columns: {missing_cols}"
top10_df = top10_df.rename(columns=B_COLS)

# Same document text the embedding experiment used: law name + article number + text
top10_df["doc_text"] = (
    "اسم النظام: " + top10_df["law_name"] +
    "\nرقم المادة: " + top10_df["article_number"] +
    "\nالنص القانوني: " + top10_df["text"]
)
top10_df["retrieval_rank"] = top10_df["retrieval_rank"].astype(int)

# Keep B's query order, candidates in B's ranking order
query_order = list(pd.unique(top10_df["query_id"]))
top10_df["_q"] = top10_df["query_id"].map({q: i for i, q in enumerate(query_order)})
top10_df = (top10_df.sort_values(["_q", "retrieval_rank"], kind="stable")
                    .drop(columns="_q").reset_index(drop=True))

# Retrieval method is read from B, not written by hand
methods_in_file = top10_df["selected_retrieval"].str.strip().unique()
assert len(methods_in_file) == 1, f"Expected one selected_retrieval, found {list(methods_in_file)}"
SELECTED_RETRIEVAL = methods_in_file[0]

# Checks: 10 candidates per query, ranks 1..10, no duplicate articles
per_q = top10_df.groupby("query_id", sort=False)
assert (per_q.size() == TOP_K).all(), "Every query must have exactly 10 candidates"
assert per_q["retrieval_rank"].apply(lambda r: list(r) == list(range(1, TOP_K + 1))).all()
assert (per_q["doc_idx"].nunique() == TOP_K).all(), "Duplicate document inside a query's Top-10"
for col in ["query", "query_type", "gold_law", "gold_article"]:
    assert (per_q[col].nunique() == 1).all(), f"'{col}' changes inside a query"

print("Selected retrieval (from B):", SELECTED_RETRIEVAL)
print("Queries:", len(query_order), "| rows:", len(top10_df))
print("Validation passed")
print()
print(top10_df.loc[0, "doc_text"][:300])

Selected retrieval (from B): Dense
Queries: 60 | rows: 600
Validation passed

اسم النظام: نظام التكاليف القضائية
رقم المادة: المادة السابعةعشرة
النص القانوني: المادة السابعةعشرة
مع مراعاة ما تقضي به الأنظمة والمعاهدات والاتفاقيات الدولية التي تكون المملكة طرفاًء لا تفرض التكاليف القضائية على الفئاتالآتية:
١.المسجونون والموقوفون وقت استحقاق التكاليف القضائية في قضايا مالية غير


In [7]:
def norm_key(x):
    # ignore spaces: "المادة الثالثة عشرة" == "المادة الثالثةعشرة"
    return re.sub(r"\s+", "", str(x))

top10_df["is_gold"] = (
    (top10_df["law_name"].map(norm_key) == top10_df["gold_law"].map(norm_key))
    & (top10_df["article_number"].map(norm_key) == top10_df["gold_article"].map(norm_key))
)

queries = []
for qid, g in top10_df.groupby("query_id", sort=False):
    g = g.reset_index(drop=True)
    gold_pos = np.flatnonzero(g["is_gold"].to_numpy())
    queries.append({
        "query_id": qid,
        "query_type": g.at[0, "query_type"],
        "query": g.at[0, "query"],
        "gold_law": g.at[0, "gold_law"],
        "gold_article": g.at[0, "gold_article"],
        "rows": g,
        "doc_texts": g["doc_text"].tolist(),
        "is_gold": g["is_gold"].to_numpy(),
        "retrieval_gold_rank": int(gold_pos[0]) + 1 if len(gold_pos) else None,
    })

n_q = len(queries)
not_in_top10 = [q for q in queries if q["retrieval_gold_rank"] is None]
CEILING_HITS = n_q - len(not_in_top10)

print(f"Gold found in {SELECTED_RETRIEVAL} Top-10: {CEILING_HITS}/{n_q} "
      f"-> Recall@10 = {CEILING_HITS / n_q:.3f}  (max any reranker can reach)")
for q in not_in_top10:
    print("  not in Top-10:", q["query_id"], "|", q["gold_law"], "|", q["gold_article"])

Gold found in Dense Top-10: 58/60 -> Recall@10 = 0.967  (max any reranker can reach)
  not in Top-10: Q051 | اللائحة التنفيذية لنظام المحاكم التجارية | المادة الخامسة والستون
  not in Top-10: Q053 | اللائحة التنفيذية لنظام المحاكم التجارية | المادة التاسعة والستون


In [8]:
def gold_rank_after_rerank(order, is_gold):
    """order: new order of the 10 candidates; is_gold: flags in B's order."""
    hits = np.flatnonzero(np.asarray(is_gold)[np.asarray(order)])
    return int(hits[0]) + 1 if len(hits) else None


def metric_values(gold_rank):
    # gold not in Top-10 -> rank None, everything = 0
    hit = gold_rank is not None and gold_rank <= TOP_K
    return {
        "Recall@3":  int(hit and gold_rank <= 3),
        "Recall@5":  int(hit and gold_rank <= 5),
        "Recall@10": int(hit),
        "RR@10":     1.0 / gold_rank if hit else 0.0,
    }

print("Metrics ready")

Metrics ready


In [9]:
from FlagEmbedding import FlagReranker
import time

BGE_NAME = "BAAI/bge-reranker-v2-m3"
BGE_MAX_LENGTH = 1024

bge_reranker = FlagReranker(BGE_NAME, use_fp16=True)


def rerank_bge(query, doc_texts):
    pairs = [[query, d] for d in doc_texts]
    scores = np.atleast_1d(np.asarray(
        bge_reranker.compute_score(pairs, max_length=BGE_MAX_LENGTH), dtype=float))
    order = np.argsort(-scores, kind="stable")   # ties keep B's order
    return order, scores


_ = rerank_bge("ما حقوق العامل؟", queries[0]["doc_texts"])   # warm-up
print("BGE reranker loaded")

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

BGE reranker loaded


In [10]:
rerank_runs = {}   # results of each reranker, per query


def run_reranker(name, rerank_fn):
    out = {}
    for q in queries:
        start = time.perf_counter()
        order, scores = rerank_fn(q["query"], q["doc_texts"])
        elapsed = time.perf_counter() - start
        assert sorted(order.tolist()) == list(range(TOP_K))
        out[q["query_id"]] = {"order": order, "scores": scores, "time_s": elapsed}
    rerank_runs[name] = out

    ranks = [gold_rank_after_rerank(out[q["query_id"]]["order"], q["is_gold"]) for q in queries]
    m = pd.DataFrame([metric_values(r) for r in ranks]).mean()
    print(f"{name:5} | R@3 = {m['Recall@3']:.3f} | R@5 = {m['Recall@5']:.3f} | "
          f"R@10 = {m['Recall@10']:.3f} | MRR@10 = {m['RR@10']:.3f} | "
          f"avg time = {np.mean([v['time_s'] for v in out.values()]):.3f} s")


run_reranker("BGE", rerank_bge)

BGE   | R@3 = 0.900 | R@5 = 0.933 | R@10 = 0.967 | MRR@10 = 0.823 | avg time = 0.030 s


In [11]:
import transformers
print("transformers:", transformers.__version__)

from transformers import AutoTokenizer, AutoModelForCausalLM

QWEN_NAME = "Qwen/Qwen3-Reranker-0.6B"
QWEN_MAX_LENGTH = 1024   # same limit as BGE
QWEN_INSTRUCTION = "Given a Saudi legal question, retrieve the legal article that answers it"

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME, padding_side="left")
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_NAME, dtype=torch.float16
).cuda().eval()

token_false_id = qwen_tokenizer.convert_tokens_to_ids("no")
token_true_id  = qwen_tokenizer.convert_tokens_to_ids("yes")

prefix = ("<|im_start|>system\nJudge whether the Document meets the requirements "
          "based on the Query and the Instruct provided. Note that the answer can "
          "only be \"yes\" or \"no\".<|im_end|>\n<|im_start|>user\n")
suffix = "<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
prefix_tokens = qwen_tokenizer.encode(prefix, add_special_tokens=False)
suffix_tokens = qwen_tokenizer.encode(suffix, add_special_tokens=False)


def format_instruction(query, doc):
    return f"<Instruct>: {QWEN_INSTRUCTION}\n<Query>: {query}\n<Document>: {doc}"


def process_inputs(pairs):
    inputs = qwen_tokenizer(
        pairs, padding=False, truncation="longest_first",
        return_attention_mask=False,
        max_length=QWEN_MAX_LENGTH - len(prefix_tokens) - len(suffix_tokens),
    )
    for i, ids in enumerate(inputs["input_ids"]):
        inputs["input_ids"][i] = prefix_tokens + ids + suffix_tokens
    inputs = qwen_tokenizer.pad(inputs, padding=True, return_tensors="pt")
    return {k: v.to(qwen_model.device) for k, v in inputs.items()}


@torch.no_grad()
def qwen_scores(pairs):
    logits = qwen_model(**process_inputs(pairs)).logits[:, -1, :]
    two = torch.stack([logits[:, token_false_id], logits[:, token_true_id]], dim=1)
    return torch.nn.functional.log_softmax(two, dim=1)[:, 1].exp().tolist()


def rerank_qwen(query, doc_texts):
    pairs = [format_instruction(query, d) for d in doc_texts]
    scores = np.asarray(qwen_scores(pairs), dtype=float)
    order = np.argsort(-scores, kind="stable")   # ties keep B's order
    return order, scores


_ = rerank_qwen("ما حقوق العامل؟", queries[0]["doc_texts"])   # warm-up
print("Qwen reranker loaded")

transformers: 5.22.1


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Qwen reranker loaded


In [12]:
run_reranker("Qwen", rerank_qwen)

Qwen  | R@3 = 0.867 | R@5 = 0.933 | R@10 = 0.967 | MRR@10 = 0.813 | avg time = 0.052 s


In [13]:
RERANKERS = ["BGE", "Qwen"]
METHODS = ["Retrieval"] + RERANKERS     # "Retrieval" = B's Top-10 in B's original order
RERANKER_MODEL = {"BGE": BGE_NAME, "Qwen": QWEN_NAME}
METHOD_LABEL = {"Retrieval": f"{SELECTED_RETRIEVAL} (B Top-10)",
                "BGE": "BGE-Reranker-v2-m3", "Qwen": "Qwen3-Reranker-0.6B"}

rows = []
for q in queries:
    qid = q["query_id"]
    row = {k: q[k] for k in ["query_id", "query_type", "query", "gold_law", "gold_article"]}
    row["selected_retrieval"] = SELECTED_RETRIEVAL

    ranks = {"Retrieval": q["retrieval_gold_rank"]}
    for m in RERANKERS:
        ranks[m] = gold_rank_after_rerank(rerank_runs[m][qid]["order"], q["is_gold"])

    for m in METHODS:
        row[f"{m}_Gold_Rank"] = ranks[m]
        for k, v in metric_values(ranks[m]).items():
            row[f"{m}_{k}"] = v
        if m in RERANKERS:
            row[f"{m}_Time_s"] = rerank_runs[m][qid]["time_s"]

    for m in RERANKERS:
        run = rerank_runs[m][qid]
        for r in range(3):
            c = q["rows"].iloc[run["order"][r]]
            row[f"{m}_Top{r+1}_Law"] = c["law_name"]
            row[f"{m}_Top{r+1}_Article"] = c["article_number"]
            row[f"{m}_Top{r+1}_Score"] = float(run["scores"][run["order"][r]])
    rows.append(row)

results_df = pd.DataFrame(rows)
for m in METHODS:
    results_df[f"{m}_Gold_Rank"] = results_df[f"{m}_Gold_Rank"].astype("Int64")

# Reranking inside the Top-10 must not change which queries have the gold in the Top-10
for m in RERANKERS:
    assert results_df[f"{m}_Recall@10"].equals(results_df["Retrieval_Recall@10"])

display(results_df.head())

,query_id,query_type,query,gold_law,gold_article,selected_retrieval,Retrieval_Gold_Rank,Retrieval_Recall@3,Retrieval_Recall@5,Retrieval_Recall@10,...,BGE_Top3_Score,Qwen_Top1_Law,Qwen_Top1_Article,Qwen_Top1_Score,Qwen_Top2_Law,Qwen_Top2_Article,Qwen_Top2_Score,Qwen_Top3_Law,Qwen_Top3_Article,Qwen_Top3_Score
0,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,3,1,1,1,...,2.070312,نظام التكاليف القضائية,المادة الثانية,0.996582,نظام التكاليف القضائية,المادة الثامنة,0.983398,نظام التكاليف القضائية,المادة الأولى,0.981934
1,Q002,formal_paraphrase,ما الحالات المستثناة من تطبيق نظام التكاليف ال...,نظام التكاليف القضائية,المادة الثانية,Dense,5,0,1,1,...,0.802734,نظام التكاليف القضائية,المادة الثانية,0.955566,نظام التكاليف القضائية,المادة السادسةعشرة,0.928223,نظام التكاليف القضائية,المادة السابعةعشرة,0.855957
2,Q003,informal,وش القضايا اللي ما عليها تكاليف قضائية؟,نظام التكاليف القضائية,المادة الثانية,Dense,9,0,0,1,...,0.278809,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثالثة,0.903320,نظام التكاليف القضائية,المادة السادسةعشرة,0.733887,نظام التكاليف القضائية,المادة السادسة,0.699219
3,Q004,formal_direct,ما الحد الأعلى للتكاليف القضائية المفروضة على ...,نظام التكاليف القضائية,المادة الثالثة,Dense,1,1,1,1,...,1.506836,نظام التكاليف القضائية,المادة الثالثة,0.992188,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثانية,0.970703,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثالثة,0.962402
4,Q005,formal_paraphrase,كيف يتم تحديد قيمة التكاليف القضائية بالنسبة إ...,نظام التكاليف القضائية,المادة الثالثة,Dense,2,1,1,1,...,3.476562,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الثانية,0.997070,اللائحة التنفيذية لنظام التكاليف القضائية,المادة الخامسة,0.973633,نظام التكاليف القضائية,المادة الثالثة,0.967773


In [14]:
def summary_row(m, g):
    return {
        "Method": METHOD_LABEL[m],
        "n": len(g),
        "Hits@3": int(g[f"{m}_Recall@3"].sum()),
        "Recall@3": g[f"{m}_Recall@3"].mean(),
        "Hits@5": int(g[f"{m}_Recall@5"].sum()),
        "Recall@5": g[f"{m}_Recall@5"].mean(),
        "Hits@10": int(g[f"{m}_Recall@10"].sum()),
        "Recall@10": g[f"{m}_Recall@10"].mean(),
        "MRR@10": g[f"{m}_RR@10"].mean(),
        "Avg_Rerank_Time_s": g[f"{m}_Time_s"].mean() if m in RERANKERS else np.nan,
    }

overall_df = pd.DataFrame([summary_row(m, results_df) for m in METHODS])
display(overall_df.round(3))
print("Recall@10 is the same for all rows: it is the retrieval ceiling from B.")

,Method,n,Hits@3,Recall@3,Hits@5,Recall@5,Hits@10,Recall@10,MRR@10,Avg_Rerank_Time_s
0,Dense (B Top-10),60,50,0.833,55,0.917,58,0.967,0.734,NaN
1,BGE-Reranker-v2-m3,60,54,0.900,56,0.933,58,0.967,0.823,0.030
2,Qwen3-Reranker-0.6B,60,52,0.867,56,0.933,58,0.967,0.813,0.052


Recall@10 is the same for all rows: it is the retrieval ceiling from B.


In [15]:
type_rows = []
for qtype, g in results_df.groupby("query_type"):
    for m in METHODS:
        type_rows.append({"query_type": qtype, **summary_row(m, g)})
by_type_df = pd.DataFrame(type_rows).drop(columns="Avg_Rerank_Time_s")
display(by_type_df.round(3))
print("Note: ~20 queries per type, so a 0.05 difference = 1 query. Descriptive only.")

,query_type,Method,n,Hits@3,Recall@3,Hits@5,Recall@5,Hits@10,Recall@10,MRR@10
0,formal_direct,Dense (B Top-10),20,20,1.00,20,1.00,20,1.00,0.833
1,formal_direct,BGE-Reranker-v2-m3,20,20,1.00,20,1.00,20,1.00,0.917
2,formal_direct,Qwen3-Reranker-0.6B,20,19,0.95,20,1.00,20,1.00,0.938
3,formal_paraphrase,Dense (B Top-10),20,16,0.80,18,0.90,19,0.95,0.678
4,formal_paraphrase,BGE-Reranker-v2-m3,20,17,0.85,18,0.90,19,0.95,0.753
5,formal_paraphrase,Qwen3-Reranker-0.6B,20,18,0.90,19,0.95,19,0.95,0.821
6,informal,Dense (B Top-10),20,14,0.70,17,0.85,19,0.95,0.691
7,informal,BGE-Reranker-v2-m3,20,17,0.85,18,0.90,19,0.95,0.800
8,informal,Qwen3-Reranker-0.6B,20,15,0.75,17,0.85,19,0.95,0.679


Note: ~20 queries per type, so a 0.05 difference = 1 query. Descriptive only.


In [16]:
COMPARISONS = [("BGE", "Retrieval"), ("Qwen", "Retrieval"), ("BGE", "Qwen")]


def safe_rank(x):
    return 10**9 if pd.isna(x) else int(x)


def win_tie_loss(a, b):
    w = t = l = 0
    for _, r in results_df.iterrows():
        ra, rb = safe_rank(r[f"{a}_Gold_Rank"]), safe_rank(r[f"{b}_Gold_Rank"])
        if ra < rb: w += 1
        elif ra == rb: t += 1
        else: l += 1
    return {"Comparison": f"{METHOD_LABEL[a]} vs {METHOD_LABEL[b]}",
            "First better": w, "Same": t, "First worse": l}

wtl_df = pd.DataFrame([win_tie_loss(a, b) for a, b in COMPARISONS])
display(wtl_df)

,Comparison,First better,Same,First worse
0,BGE-Reranker-v2-m3 vs Dense (B Top-10),18,36,6
1,Qwen3-Reranker-0.6B vs Dense (B Top-10),13,41,6
2,BGE-Reranker-v2-m3 vs Qwen3-Reranker-0.6B,11,41,8


In [17]:
rng = np.random.default_rng(42)
N_BOOT = 5000
TEST_METRICS = [("Recall@3", "Recall@3"), ("Recall@5", "Recall@5"), ("MRR@10", "RR@10")]


def paired_bootstrap(a_vals, b_vals):
    a, b = np.asarray(a_vals, float), np.asarray(b_vals, float)
    n = len(a)
    diffs = np.empty(N_BOOT)
    for i in range(N_BOOT):
        idx = rng.integers(0, n, n)
        diffs[i] = a[idx].mean() - b[idx].mean()
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return a.mean() - b.mean(), lo, hi

ci_rows = []
for a, b in COMPARISONS:
    for metric, suf in TEST_METRICS:
        d, lo, hi = paired_bootstrap(results_df[f"{a}_{suf}"], results_df[f"{b}_{suf}"])
        ci_rows.append({"Comparison": f"{a} - {b}", "Metric": metric,
                        "Diff": d, "CI Lower": lo, "CI Upper": hi,
                        "Crosses 0": lo <= 0 <= hi})
ci_df = pd.DataFrame(ci_rows)
display(ci_df.round(3))
print("Recall@10 is not tested: it is identical for all methods.")

,Comparison,Metric,Diff,CI Lower,CI Upper,Crosses 0
0,BGE - Retrieval,Recall@3,0.067,0.017,0.133,False
1,BGE - Retrieval,Recall@5,0.017,0.000,0.050,True
2,BGE - Retrieval,MRR@10,0.089,0.018,0.161,False
3,Qwen - Retrieval,Recall@3,0.033,-0.050,0.117,True
4,Qwen - Retrieval,Recall@5,0.017,-0.033,0.067,True
5,Qwen - Retrieval,MRR@10,0.079,0.013,0.148,False
6,BGE - Qwen,Recall@3,0.033,-0.050,0.117,True
7,BGE - Qwen,Recall@5,0.000,-0.050,0.050,True
8,BGE - Qwen,MRR@10,0.011,-0.060,0.082,True


Recall@10 is not tested: it is identical for all methods.


In [18]:
from scipy.stats import binomtest
rng_perm = np.random.default_rng(42)
N_PERM = 100_000


def mcnemar_exact(a, b):
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    first_only, second_only = int((a & ~b).sum()), int((~a & b).sum())
    n = first_only + second_only
    p = 1.0 if n == 0 else binomtest(first_only, n, 0.5).pvalue
    return first_only, second_only, p


def permutation_test(a, b):
    d = np.asarray(a, float) - np.asarray(b, float)
    signs = rng_perm.choice([-1.0, 1.0], size=(N_PERM, len(d)))
    perm = (signs * d).mean(axis=1)
    p = (np.sum(np.abs(perm) >= abs(d.mean()) - 1e-12) + 1) / (N_PERM + 1)
    return int((d > 0).sum()), int((d < 0).sum()), p


def holm(pvals):
    p = np.asarray(pvals, float); m = len(p)
    adj, running = np.empty(m), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, min(1.0, (m - rank) * p[i]))
        adj[i] = running
    return adj

test_rows = []
for a, b in COMPARISONS:
    for metric, suf in TEST_METRICS:
        x, y = results_df[f"{a}_{suf}"], results_df[f"{b}_{suf}"]
        if metric.startswith("MRR"):
            fo, so, p = permutation_test(x, y); test = "Paired permutation"
        else:
            fo, so, p = mcnemar_exact(x, y); test = "Exact McNemar"
        ci = ci_df[(ci_df["Comparison"] == f"{a} - {b}") & (ci_df["Metric"] == metric)].iloc[0]
        test_rows.append({
            "Comparison": f"{a} - {b}", "Metric": metric, "Diff": x.mean() - y.mean(),
            "CI Lower": ci["CI Lower"], "CI Upper": ci["CI Upper"], "Test": test,
            "First better (queries)": fo, "Second better (queries)": so, "p-value": p,
        })

sig_df = pd.DataFrame(test_rows)
sig_df["p (Holm)"] = holm(sig_df["p-value"])
sig_df["Significant (Holm, 0.05)"] = sig_df["p (Holm)"] < 0.05
display(sig_df.round(3))

,Comparison,Metric,Diff,CI Lower,CI Upper,Test,First better (queries),Second better (queries),p-value,p (Holm),"Significant (Holm, 0.05)"
0,BGE - Retrieval,Recall@3,0.067,0.017,0.133,Exact McNemar,4,0,0.125,0.875,False
1,BGE - Retrieval,Recall@5,0.017,0.000,0.050,Exact McNemar,1,0,1.000,1.000,False
2,BGE - Retrieval,MRR@10,0.089,0.018,0.161,Paired permutation,18,6,0.017,0.154,False
3,Qwen - Retrieval,Recall@3,0.033,-0.050,0.117,Exact McNemar,4,2,0.688,1.000,False
4,Qwen - Retrieval,Recall@5,0.017,-0.033,0.067,Exact McNemar,2,1,1.000,1.000,False
5,Qwen - Retrieval,MRR@10,0.079,0.013,0.148,Paired permutation,13,6,0.028,0.222,False
6,BGE - Qwen,Recall@3,0.033,-0.050,0.117,Exact McNemar,4,2,0.688,1.000,False
7,BGE - Qwen,Recall@5,0.000,-0.050,0.050,Exact McNemar,1,1,1.000,1.000,False
8,BGE - Qwen,MRR@10,0.011,-0.060,0.082,Paired permutation,11,8,0.770,1.000,False


In [19]:
transformers.logging.set_verbosity_error()
bge_rr_tokenizer = AutoTokenizer.from_pretrained(BGE_NAME)

trunc_rows = []
for q in queries:
    for doc, gold in zip(q["doc_texts"], q["is_gold"]):
        trunc_rows.append({
            "query_id": q["query_id"],
            "is_gold": bool(gold),
            "BGE_tokens": len(bge_rr_tokenizer(q["query"], doc)["input_ids"]),
            "Qwen_tokens": len(prefix_tokens) + len(suffix_tokens)
                           + len(qwen_tokenizer(format_instruction(q["query"], doc))["input_ids"]),
        })
trunc_df = pd.DataFrame(trunc_rows)
trunc_df["BGE_truncated"] = trunc_df["BGE_tokens"] > BGE_MAX_LENGTH
trunc_df["Qwen_truncated"] = trunc_df["Qwen_tokens"] > QWEN_MAX_LENGTH

trunc_summary = pd.DataFrame([{
    "Model": m,
    "Pairs": len(trunc_df),
    "Truncated pairs": int(trunc_df[f"{m}_truncated"].sum()),
    "Truncated %": round(100 * trunc_df[f"{m}_truncated"].mean(), 1),
    "Gold pairs truncated": int(trunc_df.loc[trunc_df["is_gold"], f"{m}_truncated"].sum()),
    "Median tokens": int(trunc_df[f"{m}_tokens"].median()),
} for m in RERANKERS])
display(trunc_summary)

,Model,Pairs,Truncated pairs,Truncated %,Gold pairs truncated,Median tokens
0,BGE,600,0,0.0,0,157
1,Qwen,600,0,0.0,0,267


In [20]:
SELECTION_ORDER = ["Recall@3", "MRR@10", "Recall@5"]   # higher is better; last tie-break: faster

sel_df = pd.DataFrame([{
    "Reranker": m,
    "Model": RERANKER_MODEL[m],
    "Recall@3": results_df[f"{m}_Recall@3"].mean(),
    "MRR@10": results_df[f"{m}_RR@10"].mean(),
    "Recall@5": results_df[f"{m}_Recall@5"].mean(),
    "Avg_Rerank_Time_s": results_df[f"{m}_Time_s"].mean(),
} for m in RERANKERS])

sort_keys = SELECTION_ORDER + ["Avg_Rerank_Time_s"]
sel_df = sel_df.sort_values(sort_keys, ascending=[False] * len(SELECTION_ORDER) + [True],
                            key=lambda s: s.round(10), kind="stable").reset_index(drop=True)
sel_df.insert(0, "Selection_Rank", range(1, len(sel_df) + 1))

BEST_RERANKER = sel_df.at[0, "Reranker"]
RUNNER_UP = sel_df.at[1, "Reranker"]
DECIDED_BY = next((k for k in sort_keys
                   if round(sel_df.at[0, k], 10) != round(sel_df.at[1, k], 10)), "full tie")

display(sel_df.round(3))
if DECIDED_BY == "full tie":
    print(f"Best reranker: {METHOD_LABEL[BEST_RERANKER]} (full tie)")
else:
    print(f"Best reranker: {METHOD_LABEL[BEST_RERANKER]}  "
          f"(decided by {DECIDED_BY}: {sel_df.at[0, DECIDED_BY]:.3f} vs {sel_df.at[1, DECIDED_BY]:.3f})")

pair = sorted([BEST_RERANKER, RUNNER_UP], key=RERANKERS.index)
head2head = sig_df[sig_df["Comparison"] == f"{pair[0]} - {pair[1]}"]
if head2head["Significant (Holm, 0.05)"].any():
    print("The gap between the two rerankers is statistically significant on at least one metric.")
else:
    print("Note: the gap between the two rerankers is NOT statistically significant "
          "on this test set — report it as a choice based on the observed numbers.")

,Selection_Rank,Reranker,Model,Recall@3,MRR@10,Recall@5,Avg_Rerank_Time_s
0,1,BGE,BAAI/bge-reranker-v2-m3,0.900,0.823,0.933,0.030
1,2,Qwen,Qwen/Qwen3-Reranker-0.6B,0.867,0.813,0.933,0.052


Best reranker: BGE-Reranker-v2-m3  (decided by Recall@3: 0.900 vs 0.867)
Note: the gap between the two rerankers is NOT statistically significant on this test set — report it as a choice based on the observed numbers.


In [22]:
STAGE3_TOP_N = 3
STAGE3_PATH = "/content/SANAD_Experiment_C_Top3_for_Stage3.csv"

out_rows = []
for q in queries:
    run = rerank_runs[BEST_RERANKER][q["query_id"]]
    for new_rank, pos in enumerate(run["order"][:STAGE3_TOP_N], start=1):
        c = q["rows"].iloc[pos]
        out_rows.append({
            "query_id": q["query_id"],
            "query_type": q["query_type"],
            "query": q["query"],
            "gold_law": q["gold_law"],          # for evaluation only
            "gold_article": q["gold_article"],  # for evaluation only
            "selected_retrieval": SELECTED_RETRIEVAL,
            "selected_reranker": RERANKER_MODEL[BEST_RERANKER],
            "rerank_rank": new_rank,
            "rerank_score": float(run["scores"][pos]),
            "retrieval_rank": int(c["retrieval_rank"]),
            "doc_idx": int(c["doc_idx"]),
            "law_name": c["law_name"],
            "article_number": c["article_number"],
            "doc_text": c["doc_text"],
            "is_gold": int(q["is_gold"][pos]),  # for evaluation only
        })

stage3_df = pd.DataFrame(out_rows)
assert len(stage3_df) == STAGE3_TOP_N * n_q
gold_in_top3 = stage3_df.groupby("query_id", sort=False)["is_gold"].max()
assert np.isclose(gold_in_top3.mean(), results_df[f"{BEST_RERANKER}_Recall@3"].mean())

stage3_df.to_csv(STAGE3_PATH, index=False, encoding="utf-8-sig")
print("Saved:", STAGE3_PATH)
print(f"Reranker: {RERANKER_MODEL[BEST_RERANKER]} | rows: {len(stage3_df)} "
      f"| gold in Top-3: {int(gold_in_top3.sum())}/{n_q}")
display(stage3_df.head(6))

Saved: /content/SANAD_Experiment_C_Top3_for_Stage3.csv
Reranker: BAAI/bge-reranker-v2-m3 | rows: 180 | gold in Top-3: 54/60


,query_id,query_type,query,gold_law,gold_article,selected_retrieval,selected_reranker,rerank_rank,rerank_score,retrieval_rank,doc_idx,law_name,article_number,doc_text,is_gold
0,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,1,4.171875,3,1078,نظام التكاليف القضائية,المادة الثانية,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,1
1,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,2,4.097656,4,816,نظام التكاليف القضائية,المادة الثامنة,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,0
2,Q001,formal_direct,ما الدعاوى والطلبات التي لا تسري عليها أحكام ا...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,3,2.070312,1,967,نظام التكاليف القضائية,المادة السابعةعشرة,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,0
3,Q002,formal_paraphrase,ما الحالات المستثناة من تطبيق نظام التكاليف ال...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,1,3.267578,1,967,نظام التكاليف القضائية,المادة السابعةعشرة,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,0
4,Q002,formal_paraphrase,ما الحالات المستثناة من تطبيق نظام التكاليف ال...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,2,1.934570,4,371,نظام التكاليف القضائية,المادة السادسةعشرة,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,0
5,Q002,formal_paraphrase,ما الحالات المستثناة من تطبيق نظام التكاليف ال...,نظام التكاليف القضائية,المادة الثانية,Dense,BAAI/bge-reranker-v2-m3,3,0.802734,3,2305,نظام التكاليف القضائية,المادة العاشرة,اسم النظام: نظام التكاليف القضائية\nرقم المادة...,0


In [23]:
trunc_note = "; ".join(
    f"{r['Model']}: {r['Truncated pairs']} of {r['Pairs']} pairs truncated "
    f"(gold pairs: {r['Gold pairs truncated']})" for _, r in trunc_summary.iterrows())

settings_df = pd.DataFrame({
    "Setting": ["Input file", "Selected retrieval (from B)", "Queries", "Candidates",
                "Retrieval ceiling (Recall@10)", "Rerankers", "Max length (tokens)", "Truncation",
                "Qwen instruction", "Precision", "GPU", "Time measured", "Gold matching",
                "Metrics", "Significance tests", "Selection rule", "Selected reranker",
                "Stage 3 file"],
    "Value": [
        os.path.basename(TOP10_PATH),
        SELECTED_RETRIEVAL,
        n_q,
        f"Top-{TOP_K} from B, identical for both rerankers; only reordered, nothing appended after rank {TOP_K}",
        f"{CEILING_HITS}/{n_q} = {CEILING_HITS / n_q:.3f}",
        f"{BGE_NAME} vs {QWEN_NAME}",
        f"BGE {BGE_MAX_LENGTH}, Qwen {QWEN_MAX_LENGTH}",
        trunc_note,
        QWEN_INSTRUCTION,
        "fp16",
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        f"Reranking only, one batch of {TOP_K} pairs per query; retrieval time belongs to Experiment B",
        "Whitespace-insensitive on law name + article number, inside the Top-10 (same rule as B)",
        "Recall@3/5/10 and MRR@10 inside the Top-10; gold not in Top-10 -> rank None, RR@10 = 0; "
        "Recall@10 is identical for all methods (retrieval ceiling)",
        f"Exact McNemar (Recall), paired permutation (MRR@10), Holm-adjusted over {len(sig_df)} tests; "
        "bootstrap 95% CI",
        " > ".join(SELECTION_ORDER) + " > faster",
        f"{RERANKER_MODEL[BEST_RERANKER]} (decided by {DECIDED_BY})",
        f"{os.path.basename(STAGE3_PATH)} (Top-{STAGE3_TOP_N} per query)",
    ],
})

OUTPUT_XLSX = "/content/SANAD_Experiment_C_Reranker_Comparison.xlsx"
PER_QUERY_CSV = "/content/SANAD_Experiment_C_Per_Query.csv"

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as w:
    settings_df.to_excel(w, sheet_name="Settings", index=False)
    overall_df.to_excel(w, sheet_name="Overall", index=False)
    by_type_df.to_excel(w, sheet_name="By_Query_Type", index=False)
    wtl_df.to_excel(w, sheet_name="Win_Tie_Loss", index=False)
    sig_df.to_excel(w, sheet_name="Significance_Tests", index=False)
    trunc_summary.to_excel(w, sheet_name="Truncation", index=False)
    sel_df.to_excel(w, sheet_name="Reranker_Selection", index=False)
    results_df.to_excel(w, sheet_name="Per_Query", index=False)
    stage3_df.to_excel(w, sheet_name="Top3_for_Stage3", index=False)

results_df.to_csv(PER_QUERY_CSV, index=False, encoding="utf-8-sig")

display(settings_df)
print("Saved:")
for p in [OUTPUT_XLSX, PER_QUERY_CSV, STAGE3_PATH]:
    print(" ", p)

files.download(OUTPUT_XLSX)
files.download(PER_QUERY_CSV)
files.download(STAGE3_PATH)

,Setting,Value
0,Input file,SANAD_Experiment_B_Top10_for_Reranking.csv
1,Selected retrieval (from B),Dense
2,Queries,60
3,Candidates,"Top-10 from B, identical for both rerankers; o..."
4,Retrieval ceiling (Recall@10),58/60 = 0.967
5,Rerankers,BAAI/bge-reranker-v2-m3 vs Qwen/Qwen3-Reranker...
6,Max length (tokens),"BGE 1024, Qwen 1024"
7,Truncation,BGE: 0 of 600 pairs truncated (gold pairs: 0);...
8,Qwen instruction,"Given a Saudi legal question, retrieve the leg..."
9,Precision,fp16


Saved:
  /content/SANAD_Experiment_C_Reranker_Comparison.xlsx
  /content/SANAD_Experiment_C_Per_Query.csv
  /content/SANAD_Experiment_C_Top3_for_Stage3.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>